# Diamond Price Prediction — Exploratory Data Analysis

This notebook provides a comprehensive exploratory data analysis (EDA) of the diamond dataset. We examine feature distributions, correlations, outliers, and relationships between features and the target variable (price).

**Dataset**: ~193,000 diamonds with 10 features

**Target**: `price` (USD)

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded successfully.')

---
## 1. Dataset Overview

Load the gemstone dataset and inspect its structure, data types, and summary statistics.

In [ ]:
# Load dataset
df = pd.read_csv('../Notebook_Experiments/Data/gemstone.csv')
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(10)

In [ ]:
# Data types and memory usage
df.info()

In [ ]:
# Summary statistics for numerical features
df.describe().round(2)

In [ ]:
# Summary statistics for categorical features

df.describe(include=[object])

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates} ({duplicates / len(df) * 100:.2f}%)')

---
## 2. Missing Value Analysis

Check for missing values across all columns and visualize them.

In [ ]:
# Missing values count
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df)
print(f'\nTotal missing values: {missing.sum()}')

In [ ]:
# Missing value heatmap
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis', ax=ax)
ax.set_title('Missing Value Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Feature Distribution Plots

### 3.1 Numerical Features
Histograms with KDE for all numerical columns.

In [ ]:
numerical_cols = ['carat', 'depth', 'table', 'x', 'y', 'z', 'price']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    sns.histplot(data=df, x=col, kde=True, ax=axes[i], color=sns.color_palette('viridis', len(numerical_cols))[i], bins=50)
    axes[i].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.2f}')
    axes[i].axvline(df[col].median(), color='orange', linestyle='--', label=f'Median: {df[col].median():.2f}')
    axes[i].legend(fontsize=8)

# Hide unused subplots
for j in range(len(numerical_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Feature Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3.2 Categorical Features
Count plots for cut, color, and clarity.

In [ ]:
categorical_cols = ['cut', 'color', 'clarity']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(categorical_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=axes[i], palette='viridis')
    axes[i].set_title(f'Distribution of {col}', fontsize=13, fontweight='bold')
    axes[i].set_xlabel(col, fontsize=11)
    axes[i].set_ylabel('Count', fontsize=11)
    # Add count labels
    for p in axes[i].patches:
        axes[i].annotate(f'{int(p.get_height()):,}', 
                         (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='bottom', fontsize=9)

plt.suptitle('Categorical Feature Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Correlation Heatmap

Examine the linear relationships between numerical features using the Pearson correlation coefficient.

In [ ]:
# Correlation matrix
corr_matrix = df[numerical_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', mask=mask,
            square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap — Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with price
print('\nCorrelation with Price:')
print(corr_matrix['price'].sort_values(ascending=False).to_string())

---
## 5. Price vs Carat Visualization

The carat weight is typically the strongest predictor of diamond price. We visualize this relationship, colored by different categorical features.

In [ ]:
# Price vs Carat — colored by Cut (sample for performance)
sample = df.sample(n=min(10000, len(df)), random_state=42)

fig = px.scatter(sample, x='carat', y='price', color='cut',
                 title='Diamond Price vs Carat (by Cut Quality)',
                 opacity=0.6, color_discrete_sequence=px.colors.qualitative.Vivid,
                 labels={'carat': 'Carat Weight', 'price': 'Price (USD)', 'cut': 'Cut Quality'})
fig.update_layout(template='plotly_white', width=900, height=550)
fig.show()

In [ ]:
# Price vs Carat — colored by Color
fig = px.scatter(sample, x='carat', y='price', color='color',
                 title='Diamond Price vs Carat (by Color Grade)',
                 opacity=0.6, color_discrete_sequence=px.colors.qualitative.Safe,
                 labels={'carat': 'Carat Weight', 'price': 'Price (USD)', 'color': 'Color Grade'})
fig.update_layout(template='plotly_white', width=900, height=550)
fig.show()

In [ ]:
# Price vs Carat — colored by Clarity
fig = px.scatter(sample, x='carat', y='price', color='clarity',
                 title='Diamond Price vs Carat (by Clarity)',
                 opacity=0.6, color_discrete_sequence=px.colors.qualitative.Pastel,
                 labels={'carat': 'Carat Weight', 'price': 'Price (USD)', 'clarity': 'Clarity'})
fig.update_layout(template='plotly_white', width=900, height=550)
fig.show()

---
## 6. Boxplots for Outlier Detection

### 6.1 Numerical Feature Outliers
Use boxplots to identify outliers across all numerical columns.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    sns.boxplot(data=df, y=col, ax=axes[i], color=sns.color_palette('Set2')[i % 8])
    axes[i].set_title(f'{col}', fontsize=12, fontweight='bold')
    axes[i].set_ylabel(col)

# Hide unused
for j in range(len(numerical_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Outlier Detection — Boxplots for Numerical Features', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 6.2 Price Distribution by Categorical Features
How does price vary across different cut, color, and clarity grades?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Price by Cut
cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
sns.boxplot(data=df, x='cut', y='price', order=cut_order, ax=axes[0], palette='viridis')
axes[0].set_title('Price by Cut Quality', fontsize=13, fontweight='bold')

# Price by Color
color_order = ['D', 'E', 'F', 'G', 'H', 'I', 'J']
sns.boxplot(data=df, x='color', y='price', order=color_order, ax=axes[1], palette='coolwarm')
axes[1].set_title('Price by Color Grade', fontsize=13, fontweight='bold')

# Price by Clarity
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']
sns.boxplot(data=df, x='clarity', y='price', order=clarity_order, ax=axes[2], palette='Spectral')
axes[2].set_title('Price by Clarity Grade', fontsize=13, fontweight='bold')

for ax in axes:
    ax.set_ylabel('Price (USD)')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Price Distribution by Categorical Features', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Key Insights

**Findings from EDA:**

1. **Carat is the strongest predictor** of price, with a very high positive correlation
2. **Dimensions (x, y, z)** are highly correlated with carat and price — larger diamonds cost more
3. **Depth and table** have weaker correlation with price
4. **Cut quality** does not show a simple linear relationship with price (Premium/Fair can be expensive due to higher carat weights)
5. **Price distribution** is right-skewed — most diamonds are in the lower price range
6. **Outliers** exist in dimensions (x, y, z) — some diamonds have zero values, indicating data quality issues
7. The dataset has **no missing values**, making it ready for modeling
8. **Ideal cut** is the most common category, while **J color** and **I1 clarity** are the rarest